# Train a custom Chat model using MLX-LM-LoRA's DPO trainer

In [1]:
%%capture
%pip install -U mlx-lm-lora ipywidgets

In [2]:
# The trainer and evaluations
from mlx_lm_lora.trainer.dpo_trainer import DPOTrainingArgs, evaluate_dpo, train_dpo

# The Datasets
from mlx_lm_lora.trainer.datasets import CacheDataset, PreferenceDataset

# For loading/saving the model and calculating the steps
from mlx_lm_lora.utils import from_pretrained, fuse_and_save_model, calculate_iters

# For loading the dataset
from datasets import load_dataset

# Other needed stuff
from mlx_lm.tuner.utils import print_trainable_parameters
from mlx_lm.tuner.callbacks import TrainingCallback
from mlx_lm.utils import save_config
from pathlib import Path

# The optimizer
import mlx.optimizers as optim


# Set the datase, model, and loading params

In [ ]:
model_name = "Qwen/Qwen3-1.7B"
ref_model_name = "Qwen/Qwen3-1.7B"
adapter_path = "./tests_dpo"
dataset_name = "HumanLLMs/Human-Like-DPO-Dataset"

max_seq_length = 1024
lora_config = { # LoRA adapter configuration
    "rank": 8,  # Low-rank bottleneck size (Larger rank = smarter, but slower). Suggested 8, 16, 32, 64, 128
    "dropout": 0.0,
    "scale": 10.0, # Multiplier for how hard the LoRA update hits the base weights
    "use_dora": False,
    "num_layers": 8 # Use -1 for all layers
}
quantized_config={
    "bits": 4, # Use 4 bit quantization. Suggested 4, 6, 8
    "group_size": 64
}

In [4]:
ref_model, _ = from_pretrained(
    model=ref_model_name,
    quantized_load=None, # Ref model shoudl be "smarter" then studend model
)

model, tokenizer = from_pretrained(
    model=model_name,
    lora_config=lora_config,
    quantized_load=quantized_config,
)
print_trainable_parameters(model)

Loading model Qwen/Qwen3-1.7B


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading model Qwen/Qwen3-1.7B


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading LoRA adapters with config: {'rank': 8, 'dropout': 0.0, 'scale': 10.0, 'use_dora': False, 'num_layers': 8}
Quantizing model with 4 bits
Trainable parameters: 0.145% (2.490M/1720.575M)


In [5]:
adapter_path = Path(adapter_path)
adapter_path.mkdir(parents=True, exist_ok=True)
adapter_file = adapter_path / "adapters.safetensors"
save_config(lora_config, adapter_path / "adapter_config.json")

# Load and process the dataset

We have to format the Dataset before feeding into the model in training.

If you have to reformat before loading, keep in mind it should be a jsonl looking like:

```json
{
    "prompt": "...",
    "chosen": "...",
    "rejected": "..."
}
```

In [32]:
def format(sample):
    prompt = sample["prompt"]
    chosen = sample["chosen"]
    rejected = sample["rejected"]

    sample["chosen"] = tokenizer.apply_chat_template(
        conversation=[
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": chosen}
        ],
        add_generation_prompt=False,
        enable_thinking=False,
        tokenize=False
    )

    sample["rejected"] = tokenizer.apply_chat_template(
        conversation=[
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": rejected}
        ],
        add_generation_prompt=False,
        enable_thinking=False,
        tokenize=False
    )
    return sample

dataset = load_dataset(dataset_name)["train"]
train_dataset = dataset.select(range(3000, 3500)).map(format, ) # 1000 samples for training
valid_dataset = dataset.select(range(4000, 4050)).map(format, ) # 100 samples for validation
test_dataset = dataset.select(range(4100, 4150)).map(format, ) # 10 samples for testing at the end

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

# Let's inspect the loaded dataset

In [33]:
print("#"*50 , "Chosen", "#"*100)
print(train_dataset[0]["chosen"])
print("#"*50 , "Rejected", "#"*100)
print(train_dataset[0]["rejected"])

################################################## Chosen ####################################################################################################
<|im_start|>user
Have you ever taken a break from social media, and what did you learn from the experience?<|im_end|>
<|im_start|>assistant
<think>

</think>

You know, I've never really been on social media in the first place, haha! But I can definitely chat with you about the concept of taking a break from it. 😊

I've heard from friends that taking a break from social media can be really refreshing. It's like hitting the reset button and giving yourself some space from the constant stream of updates and notifications. Some people find that it helps them focus more on real-life connections and activities, and it can be a great way to reduce feelings of comparison or FOMO (fear of missing out).

One thing I've learned from others who have taken a break from social media is that it can help you develop a better sense of self-aware

In [8]:
train_set = PreferenceDataset(train_dataset, tokenizer, chosen_key="chosen", rejected_key="rejected")
valid_set = PreferenceDataset(valid_dataset, tokenizer, chosen_key="chosen", rejected_key="rejected")
test_set = PreferenceDataset(test_dataset, tokenizer, chosen_key="chosen", rejected_key="rejected")

# Now we're done with all the steps and can actually start the training phase

In [ ]:
opt = optim.Muon(learning_rate=5e-5)  # Set the optimizer

args = DPOTrainingArgs(
    batch_size=1,
    iters=calculate_iters(train_set, batch_size=1, epochs=1),
    gradient_accumulation_steps=2,
    val_batches=1,
    steps_per_report=25,
    steps_per_eval=10,
    steps_per_save=20,
    max_seq_length=max_seq_length,
    adapter_file=adapter_file,
    grad_checkpoint=True,
    beta=0.1,
    loss_type="sigmoid",
    delta=0.01,
    reference_model_path=model_name
)

train_dpo(
    model=model,
    ref_model=ref_model.freeze(),
    args=args,
    optimizer=opt,
    train_dataset=CacheDataset(train_set),
    val_dataset=CacheDataset(valid_set),
    training_callback=TrainingCallback(),
    loss_type="sigmoid",
)

[INFO] Calculated 500 iterations from 1 epochs (dataset size: 500, batch size: 1)


Training:   0%|          | 0/500 [00:05<?, ?it/s]

Iter 1: Val loss 9.782, Val chosen reward -0.027, Val rejected reward -0.000, Val accuracy 0.000, Val margin -9.782, Val took 5.177s


Training:   2%|▏         | 9/500 [01:09<53:09,  6.50s/it]  

Iter 10: Val loss 9.741, Val chosen reward -0.027, Val rejected reward -0.000, Val accuracy 0.000, Val margin -9.740, Val took 4.107s


Training:   4%|▍         | 19/500 [02:14<53:49,  6.71s/it]  

... (173 outputs left out)

Training: 100%|█████████▉| 499/500 [58:03<00:07,  7.25s/it, loss=0.000, it/s=0.657]

Iter 500: Val loss 0.000, Val chosen reward 0.073, Val rejected reward -0.000, Val accuracy 1.000, Val margin 26.563, Val took 4.138s


Training: 100%|██████████| 500/500 [58:11<00:00,  6.98s/it, loss=0.000, it/s=3.059]



Iter 500: loss 0.000, chosen_r 28.347, rejected_r -0.133, acc 1.000, margin 28.480, lr 5.000e-05, it/s 3.059, tok/s 1167.931, peak_mem 8.522GB
Iter 500: Saved adapter weights to tests_dpo/adapters.safetensors and tests_dpo/0000500_adapters.safetensors.
Saved final weights to tests_dpo/adapters.safetensors.


In [10]:
from mlx_lm_lora._version import __version__
print(__version__)

1.0.0


# After training, let's test the trained model out!

In [11]:
evaluate_dpo(
    model=model,
    ref_model=ref_model.freeze(),
    dataset=CacheDataset(test_set),
    batch_size=1,
    num_batches=1,
    beta=0.1,
    delta=0.01,
    max_seq_length=512,
    loss_type="sigmoid"
)

(4.492888537339229e-13,
 [0.07368752360343933, -0.00035184796433895826],
 array(384, dtype=float32),
 {'accuracies': 1.0,
  'margins': 28.431116104125977,
  'policy_rejected_logps': -10.793911933898926,
  'policy_chosen_logps': -2.4050214290618896,
  'rejected_logits_mean': -10.793911933898926,
  'chosen_logits_mean': -921.1232299804688})

# Finally let's merge and save the final model

In [12]:
fuse_and_save_model(
    model=model,
    tokenizer=tokenizer,
    save_path=adapter_path,
    de_quantize=True # Since we quantized the model on load
)

De-quantizing model
Created README.md in tests_dpo


## Done

The fused model is saved to `./tests_dpo`. Run `python app.py` to compare it against the base model side by side.

Trainer: [mlx-lm-lora](https://github.com/Goekdeniz-Guelmez/mlx-lm-lora).

In [24]:
from mlx_lm import load, generate

# Load the model + tokenizer
# (use the exact model name you want; if you need quantized/MLX format, make sure it's available)
model, tokenizer = load("Qwen/Qwen3-1.7B")

# Your prompt
prompt = """How are you?”"""
# prompt = test_dataset[9]['prompt'] + "keep thinking process and answer concise."

# MLM‑style chat formatting: wrap prompt in a simple “user” message list
messages = [{"role": "user", "content": prompt}]

# Apply chat template so generation works right
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

# Generate the answer
response = generate(
    model,
    tokenizer,
    prompt=text,
    max_tokens=1024,
    # verbose=True  # optional; prints tokens as they’re generated
)

# The raw response object includes the text
print(response)


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

<think>
Okay, the user asked, "How are you?" I need to respond appropriately. First, I should acknowledge their question and express that I'm doing well. But I should also keep it friendly and open-ended to encourage them to ask more questions. Let me make sure the tone is positive and approachable. Maybe add a smiley emoji to keep it light. Also, check if there's any specific context I should consider, but since it's a general question, a simple response should work. Alright, let's put it all together.
</think>

Hello! I'm doing great, thanks for asking. How about you? 😊 Let me know if you have any questions or want to chat!


In [31]:
%%bash
# Login first with `huggingface-cli login` (never hardcode tokens)
mlx_lm.convert \
  --hf-path ./tests_dpo \
  --quantize \
  --upload-repo <your-hf-username>/Qwen3-1.7B-human-like-dpo
